In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap

# 1. Load dataset
df = pd.read_csv('../data/raw/cleaned_credit_risk_dataset.csv')

# 2. Robust target mapping
if 'Decision' in df.columns:
    df['Decision'] = (
        df['Decision']
        .astype(str)
        .str.strip()
        .str.upper()
        .map({'APPROVE': 1, 'APPROVED': 1, '1': 1, 'REJECT': 0, 'REJECTED': 0, '0': 0})
    )

# 3. Feature Engineering (matching exact CSV column headers)
if 'Loan Amount' in df.columns and 'Annual Income' in df.columns:
    df['Debt_to_Income_Ratio'] = df['Loan Amount'] / (df['Annual Income'] + 1)

if 'Annual Income' in df.columns and 'Number of Dependents' in df.columns:
    df['Income_per_Dependent'] = df['Annual Income'] / (df['Number of Dependents'] + 1)

# 4. Prepare feature matrix X (one-hot encode categorical strings)
X = df.drop(columns=['Decision']) if 'Decision' in df.columns else df.copy()
X = pd.get_dummies(X, drop_first=True)

# 5. Load trained XGBoost model
model = xgb.XGBClassifier()
model.load_model('../models/xgboost_credit_model.json')

# 6. Define Risk Band function
def assign_risk_band(prob):
    if prob >= 0.80:
        return 'Low Risk (Auto-Approve)'
    elif prob >= 0.50:
        return 'Medium Risk (Manual Review)'
    else:
        return 'High Risk (Auto-Reject)'

# 7. Predict probabilities and assign risk bands
df['Approval_Probability'] = model.predict_proba(X)[:, 1]
df['Risk_Band'] = df['Approval_Probability'].apply(assign_risk_band)

print("--- Sample Risk Scoring Results ---")
print(df[['Approval_Probability', 'Risk_Band']].head())

# 8. SHAP Pointwise Explainer Function
explainer = shap.TreeExplainer(model)

def explain_applicant_decision(applicant_index=0):
    sample_df = X.iloc[[applicant_index]]
    prob = df.loc[applicant_index, 'Approval_Probability']
    band = df.loc[applicant_index, 'Risk_Band']
    
    # Calculate SHAP values
    shap_vals = explainer.shap_values(sample_df)
    shap_array = shap_vals[0] if isinstance(shap_vals, list) else shap_vals[0]
    
    impact_df = pd.DataFrame({
        'Feature': sample_df.columns,
        'Value': sample_df.iloc[0].values,
        'Impact': shap_array
    })
    
    impact_df['Abs_Impact'] = impact_df['Impact'].abs()
    top_factors = impact_df.sort_values(by='Abs_Impact', ascending=False).head(4)
    
    print(f"\n--- Detailed Explanation for Applicant #{applicant_index} ---")
    print(f"Risk Band: {band} (Approval Probability: {prob:.2%})")
    print("Pointwise Key Reasons:")
    
    for _, row in top_factors.iterrows():
        direction = "Increased approval chances" if row['Impact'] > 0 else "Lowered approval chances"
        print(f" • {row['Feature']} ({row['Value']}): {direction} by {abs(row['Impact']):.3f}")

# Example: Explain decision for the first applicant
explain_applicant_decision(applicant_index=0)

--- Sample Risk Scoring Results ---
   Approval_Probability                    Risk_Band
0              0.355817      High Risk (Auto-Reject)
1              0.819107      Low Risk (Auto-Approve)
2              0.725167  Medium Risk (Manual Review)
3              0.765149  Medium Risk (Manual Review)
4              0.903206      Low Risk (Auto-Approve)

--- Detailed Explanation for Applicant #0 ---
Risk Band: High Risk (Auto-Reject) (Approval Probability: 35.58%)
Pointwise Key Reasons:
 • Credit_Score (637.0): Lowered approval chances by 1.278
 • Debt_to_Income_Ratio (14.29): Increased approval chances by 0.875
 • Loan_to_Income_Ratio (41.64): Increased approval chances by 0.745
 • Previous_Defaults (1.0): Lowered approval chances by 0.374
